[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ersilia-os/ub-cedd-projects-workshop/blob/main/projects/purple/notebooks/purple_data_curation.ipynb)

# Curating HIV-1 activity data from ChEMBL

**Purple group · HIV**

This notebook curates the available bioactivity data in ChEMBL for HIV-1 (CHEMBL378) with the purpose of building an ML model

## What you will do

- Load two ChEMBL downloads for HIV-1, one of EC50 measurements and one of IC50.
- Clean the records and standardise the molecules so that the same molecule is always
  recognised as the same molecule.
- Ensure units are homogenised and decide what to do with incomplete records.
- Collapse repeated measurements into one value per molecule, and see how much the
  repeats disagree.
- Choose an activity cutoff, label every molecule as active or inactive, and
  download the result so the group can keep it.
- Build a second, smaller table for predicting potency itself, and work out how many
  molecules that costs.

## Setup

Run the cell below first. In Colab it downloads the workshop repository (including the data) and installs the packages this project needs. It takes about a minute. **Don't change it.**

In [ ]:
PROJECT = "purple"
NEEDS_GPU = False
import os, sys, shutil, subprocess
if "google.colab" in sys.modules:
    repo_dir = "/content/ub-cedd-projects-workshop"
    if not os.path.exists(repo_dir):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ersilia-os/ub-cedd-projects-workshop.git", repo_dir], check=True)
    else:
        subprocess.run(["git", "-C", repo_dir, "pull", "--ff-only"], check=True)
    os.chdir(f"{repo_dir}/projects/{PROJECT}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
has_gpu = shutil.which("nvidia-smi") is not None and subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
print(f"Python {sys.version.split()[0]} | GPU: {'yes' if has_gpu else 'no'} | Folder: {os.getcwd()}")
if NEEDS_GPU and not has_gpu:
    print("WARNING: this notebook needs a GPU. Go to Runtime > Change runtime type, choose CPU, and run this cell again.")

## 1. Load the two ChEMBL downloads

This notebook works on two files you download from ChEMBL yourself, one per type
of measurement, both for HIV-1 (its target code in ChEMBL is `CHEMBL378`).

- **EC50** is the concentration of a compound that produces half of its maximum
  effect on the virus, usually measured in infected cells.
- **IC50** is the concentration that halves whatever is being measured, often the
  growth of the virus or the activity of an enzyme.

Both answer the question "how much compound do you need?", so for both of them a
**smaller number means a better compound**. They are not the same measurement, so we
keep them apart until the last step

**To do by hand**

Search ChEMBL for `CHEMBL378`, open its list of activities, keep only one standard
type at a time, and export each as a CSV. Name them `chembl378_ec50.csv` and
`chembl378_ic50.csv`. The loader copes with either the website's column names or the
programming interface's, so it does not matter which route you take.

The first cell checks that both files are where the notebook expects them. In Colab
it opens a file picker so you can upload them, and it stops with a clear message if
they are still not there. **Colab forgets them when it disconnects**, so you will upload
them again each time you open the notebook.

In [ ]:
import os

import numpy as np
import pandas as pd
from scripts import curation

RANDOM_SEED = 42
FILES = {"EC50": "data/chembl378_ec50.csv", "IC50": "data/chembl378_ic50.csv"}

missing = [path for path in FILES.values() if not os.path.exists(path)]
if missing and "google.colab" in sys.modules:
    from google.colab import files
    os.makedirs("data", exist_ok=True)
    for name, content in files.upload().items():
        open(f"data/{name}", "wb").write(content)
    missing = [path for path in FILES.values() if not os.path.exists(path)]
if missing:
    raise FileNotFoundError(f"not found: {missing}. Add them and run this cell again.")
print("both files are in place")

Now read them in. The two files are stacked into one table, with an `endpoint`
column remembering which file each row came from.

In [ ]:
records = curation.load_downloads(FILES)
records["endpoint"].value_counts()

Each row is one measurement of one compound in one experiment. The columns we will
use are the compound's code, how it is written as a molecule, the number that was
measured, its unit, and the `standard_relation`, which says whether the number is an
exact result or only a limit.

In [ ]:
columns = ["endpoint", "molecule_chembl_id", "standard_relation",
           "standard_value", "standard_units", "canonical_smiles"]
records[columns].sample(5, random_state=RANDOM_SEED)

## 2. Clean the records

Not every row is usable. Some have no number at all, some have no structure, and some
are flagged by ChEMBL itself as suspicious. We remove those first, counting what goes
at each step so that nothing disappears silently.

ChEMBL adds a `data_validity_comment` when it doubts a record, for example when the
value is far outside the range normally seen for that kind of measurement.

In [ ]:
records["data_validity_comment"].value_counts(dropna=False).head()

We drop the flagged records, then the ones with no number or no structure. A value
of zero or less cannot be a concentration, so those go too.

In [ ]:
clean = records[records["data_validity_comment"].isna()].copy()
clean["standard_value"] = pd.to_numeric(clean["standard_value"], errors="coerce")
clean = clean[clean["standard_value"].notna() & (clean["standard_value"] > 0)]
clean = clean[clean["canonical_smiles"].notna()]
print(f"{len(records):,} records -> {len(clean):,} after cleaning")

The `standard_relation` column says how to read the number. ChEMBL writes the same
idea in several ways (`>` and `>=` and `>>`), and an empty relation means the value is
exact. We collapse them into three symbols: `=`, `>` and `<`.

In [ ]:
clean["relation"] = clean["standard_relation"].fillna("").map(curation.RELATIONS)
clean = clean[clean["relation"].isin(["=", ">", "<"])]
pd.crosstab(clean["endpoint"], clean["relation"])

Finally the units. Most values are in nanomolar, some in micromolar, and a few in
micrograms per millilitre. Anything measured in a unit we cannot turn into a
concentration, such as a percentage, has to go.

In [ ]:
usable = set(curation.UNIT_TO_NM) | curation.MASS_UNITS
print(clean.loc[~clean["standard_units"].isin(usable), "standard_units"].value_counts())
clean = clean[clean["standard_units"].isin(usable)]
clean["standard_units"].value_counts()

> **Exercise:** We dropped every record that ChEMBL flagged with a
> `data_validity_comment`. Look again at the list of reasons above. Are they all
> equally serious? Would you keep any of them, and what would you have to check
> before doing so?

## 3. Standardise the molecules

A molecule is written here as a SMILES string. The problem is that the same molecule can be written in more than
one correct SMILES, and ChEMBL often stores it as a salt, meaning the molecule plus a
counter-ion such as chloride that is not part of what is being tested.

If we do not fix this, the same compound will look like several different compounds.
The helper removes salts, rewrites every molecule in one agreed form, and computes an
**InChIKey**: a 27 character code that is identical for any two identical molecules.
From here on the InChIKey is what we mean by "a molecule".

In [ ]:
unique_smiles = clean["canonical_smiles"].unique()
structures = curation.standardize_smiles(unique_smiles)
print(f"{len(unique_smiles):,} structures read, {len(structures):,} standardised")
structures.head(3)

Now we attach the cleaned structure, the InChIKey and the molecular weight to every
measurement. The number of distinct molecules is smaller than the number of distinct
SMILES, because salts have been merged with the compound they came from.

In [ ]:
clean = clean.merge(structures, on="canonical_smiles", how="inner")
print(f"{clean['canonical_smiles'].nunique():,} SMILES -> "
      f"{clean['inchikey'].nunique():,} molecules")
clean[["smiles", "inchikey", "mw"]].head(3)

It is worth looking at the molecular weights. Most drugs weigh between about 150 and
600 g/mol. Anything much heavier is usually a peptide or a natural product, which
behaves very differently and may not suit the models we build later.

In [ ]:
clean["mw"].describe().round(0)

> **Exercise:** Decide whether to keep the very heavy molecules. Try adding a filter
> such as `clean = clean[clean["mw"] < 1000]` and see how many measurements and how
> many molecules you lose. There is no single right answer: it depends on what you
> want the model to be used for.

## 4. Put every value on the same scale

Two numbers can only be compared if they are in the same unit. We convert everything
to nanomolar. Micromolar is simply a thousand times nanomolar, but micrograms per
millilitre is a mass rather than a count of molecules, so converting it needs the
molecular weight.

In [ ]:
clean["value_nm"] = curation.to_nanomolar(
    clean["standard_value"], clean["standard_units"], clean["mw"]
)
clean = clean[clean["value_nm"].notna()]
clean.groupby("standard_units")["value_nm"].median().round(1)

Concentrations span an enormous range, from under one nanomolar to over a hundred
micromolar. That is awkward to work with, so we use **pActivity** instead: minus the
base-10 logarithm of the concentration in molar. It turns a scale where small is good
into one where large is good, and spreads the values out evenly. 1 nM becomes 9,
1 uM becomes 6, and 100 uM becomes 4.

In [ ]:
clean["pactivity"] = curation.pactivity(clean["value_nm"])
clean[["endpoint", "value_nm", "pactivity"]].sample(5, random_state=RANDOM_SEED)

A good way to check we have not made a mistake is to compare our own numbers against
ChEMBL's. ChEMBL computes the same quantity for some records and calls it
`pchembl_value`. Where both exist they should agree.

In [ ]:
check = clean[clean["pchembl_value"].notna() & (clean["relation"] == "=")]
gap = (check["pactivity"].round(2) - check["pchembl_value"]).abs()
print(f"{(gap <= 0.011).mean():.1%} of {len(check):,} comparable records agree")

## 5. Decide what to do with the incomplete measurements

Some records do not give a number but a limit. `IC50 > 50 uM` means the experiment
never reached half inhibition at the tested concentrations, so all we know is that the compound is weak.
`EC50 < 5 nM` means the opposite: it was already fully effective at the lowest
concentration tested.

These are not mistakes and they are not useless. Whether a limit is useful depends on
where we put the line between active and inactive. We need a provisional cutoff to
judge them, so we use 1 uM here and come back to that choice in section 7.

In [ ]:
CUTOFF_UM = 1.0
CUTOFF_NM = CUTOFF_UM * 1000
THRESHOLD = curation.pactivity(CUTOFF_NM)
print(f"cutoff {CUTOFF_UM} uM = {CUTOFF_NM:,.0f} nM = pActivity {THRESHOLD:.1f}")

A limit is only worth keeping when it already falls on the decided side of the
cutoff. With a 1 uM cutoff, `> 50 uM` proves the molecule is inactive and `< 5 nM`
proves it is active. But `> 100 nM` proves nothing at all: the true value could be
anywhere from just above 100 nM to completely inactive.

In [ ]:
is_bound = clean["relation"].isin([">", "<"])
decisive = ((clean["relation"] == ">") & (clean["value_nm"] >= CUTOFF_NM)) | \
           ((clean["relation"] == "<") & (clean["value_nm"] <= CUTOFF_NM))
summary = clean.assign(decisive=decisive).groupby(["endpoint", "relation"]).agg(
    records=("decisive", "size"), decisive=("decisive", "sum"))
summary.assign(share=(summary["decisive"] / summary["records"]).round(2))

> **Exercise:** The rule above throws away every limit that does not settle the
> question. Another option is to drop all limits and keep only exact numbers, which
> is simpler but loses thousands of genuine inactive compounds. A third is to keep
> `>` but not `<`, since there are far fewer of the latter. Decide which you prefer,
> and note that the answer changes if you later change the cutoff.

## 6. Collapse repeated measurements

Many molecules have been measured more than once, sometimes in dozens of different
papers. We need one value per molecule per endpoint. We take the **median** of the
exact measurements rather than the mean, because a single badly wrong number cannot
drag the median far.

In [ ]:
KEY = ["inchikey", "endpoint"]
measured = curation.summarise_replicates(clean, KEY)
measured["source"] = "measured"
print(f"{len(measured):,} molecule-endpoint pairs have at least one exact value")
measured.head(3)

Molecules that have no exact measurement at all are not lost yet: if their limits are
decisive, as discussed in section 5, we can still say whether they are active.

In [ ]:
no_exact = clean[~clean.set_index(KEY).index.isin(measured.index)]
bounded = curation.decisive_bounds(no_exact, KEY, CUTOFF_NM)
print(f"{len(bounded):,} more pairs recovered from decisive limits")
bounded["source"].value_counts()

Now we put the two groups together and attach each molecule's structure, so that we
have a single table with one row per molecule per endpoint.

In [ ]:
identity = clean.groupby(KEY).agg(smiles=("smiles", "first"),
                                  chembl_id=("molecule_chembl_id", "first"),
                                  n_records=("activity_id", "size"))
molecules = identity.join(pd.concat([measured, bounded]), how="inner").reset_index()
print(f"{len(molecules):,} rows, one per molecule and endpoint")
molecules["endpoint"].value_counts()

How much do the repeated measurements actually disagree? For every molecule measured
at least twice we look at the distance between its lowest and its highest pActivity.
A spread of 1 means the two results differ by a factor of ten.

In [ ]:
import stylia

stylia.set_format("slide")
stylia.set_style("ersilia")
nc = stylia.NamedColors()
repeats = molecules[molecules["n_exact"] >= 2]
repeats.groupby("endpoint")["spread"].describe().round(2)

The histogram makes the problem visible. The dashed line marks one log unit, that is
a tenfold disagreement between two measurements of the same molecule.

In [ ]:
fig, axs = stylia.create_figure(1, 1)
ax = axs.next()
for endpoint, color in [("EC50", nc.purple), ("IC50", nc.mint)]:
    spread = repeats.loc[repeats["endpoint"] == endpoint, "spread"]
    ax.hist(spread, bins=50, range=(0, 6), histtype="stepfilled", alpha=0.6,
            color=color, label=f"{endpoint} (median {spread.median():.2f})")
ax.axvline(1.0, color=nc.pink, linestyle="--")
ax.legend()
stylia.label(ax, xlabel="pActivity spread (highest - lowest)", ylabel="Molecules",
             title="How far apart repeated measurements are")

Disagreement only matters if it changes the answer. A molecule whose measurements all
sit above the cutoff is active whichever one you pick. A molecule whose measurements
sit on both sides of the cutoff gets its label from the rule we chose, not from the
data.

In [ ]:
always_active = repeats["pactivity_min"] >= THRESHOLD
always_inactive = repeats["pactivity_max"] < THRESHOLD
repeats.assign(verdict=np.select(
    [always_active, always_inactive], ["always active", "always inactive"],
    default="depends on the rule",
)).groupby(["endpoint", "verdict"]).size().unstack(fill_value=0)

> **Exercise:** We used the median. Another common choice is to keep the **most
> active** measurement, on the grounds that a compound is rarely made to look better
> than it is, while a bad experiment easily makes it look worse. Recompute the table
> above using `max` instead of the median and see how many labels change. Which rule
> would you defend to a chemist?

## 7. Choose a cutoff and label the molecules

To train a classifier we need each molecule labelled active or inactive, which means
drawing a line somewhere. There is no natural line. Drawing it at 1 uM is a habit, not
a law, so look at what each choice would give you before you commit.

In [ ]:
for micromolar in [0.1, 1.0, 10.0]:
    limit = curation.pactivity(micromolar * 1000)
    share = (molecules["pactivity"] >= limit).mean()
    print(f"cutoff {micromolar:>5g} uM -> {share:.1%} of molecules would be active")

The distribution shows why. Most molecules sit in a broad band between 10 uM and
10 nM, and the cutoff simply slices through it. The tall bars on the left are the
molecules we rescued from limits in section 5: laboratories test at round
concentrations such as 10 uM and 100 uM, so those pile up on a few exact values.

In [ ]:
fig, axs = stylia.create_figure(1, 1)
ax = axs.next()
for endpoint, color in [("EC50", nc.purple), ("IC50", nc.mint)]:
    values = molecules.loc[molecules["endpoint"] == endpoint, "pactivity"]
    ax.hist(values, bins=60, range=(2, 12), histtype="stepfilled", alpha=0.6,
            color=color, label=f"{endpoint} (n={len(values):,})")
ax.axvline(THRESHOLD, color=nc.pink, linestyle="--")
ax.legend()
stylia.label(ax, xlabel="pActivity", ylabel="Molecules",
             title=f"Potency distribution and the {CUTOFF_UM:g} uM cutoff")

Now the label itself. There is a trap here. A molecule whose only record is
`> 1 uM` ends up with a pActivity of exactly 6.00, and asking "is pActivity at least
6?" would call it **active**, when the limit proves the opposite. So the label has to
follow where the value came from, not just the number.

In [ ]:
molecules["activity"] = np.select(
    [molecules["source"] == "bounded inactive",
     molecules["source"] == "bounded active"],
    [0, 1],
    default=(molecules["pactivity"] >= THRESHOLD).astype(int),
)
molecules.groupby("endpoint")["activity"].value_counts().unstack()

> **Exercise:** Change `CUTOFF_UM` at the top of section 5 and run the notebook again
> from there. Watch three things: how many limits stay decisive, how balanced the two
> classes become, and how many molecules change label. A cutoff that gives a very
> uneven split makes a model much harder to train and to judge.

## 8. Merge the two endpoints

We have kept EC50 and IC50 apart on purpose. Before pooling them we should check
whether they actually tell the same story, using the molecules that were measured
both ways.

In [ ]:
wide = molecules.pivot(index="inchikey", columns="endpoint",
                       values=["pactivity", "activity"])
wide.columns = [f"{a}_{b.lower()}" for a, b in wide.columns]
both = wide.dropna(subset=["pactivity_ec50", "pactivity_ic50"])
print(f"{len(both):,} of {len(wide):,} molecules were measured both ways")

Each point below is one of those molecules. If the two endpoints agreed perfectly the
points would sit on the diagonal. The dashed lines are the cutoff, so points in the
top-left and bottom-right corners are molecules the two endpoints disagree about.

In [ ]:
agree = both["activity_ec50"] == both["activity_ic50"]
fig, axs = stylia.create_figure(1, 1, width=0.5, height=0.5)
ax = axs.next()
for mask, color, name in [(agree, nc.purple, "agree"), (~agree, nc.pink, "disagree")]:
    ax.scatter(both.loc[mask, "pactivity_ec50"], both.loc[mask, "pactivity_ic50"],
               color=color, alpha=0.5, label=f"{name} ({mask.mean():.0%})")
ax.plot([3, 11.5], [3, 11.5], color=nc.gray, linestyle=":")
ax.axvline(THRESHOLD, color=nc.pink, linestyle="--")
ax.axhline(THRESHOLD, color=nc.pink, linestyle="--")
ax.legend()
stylia.label(ax, xlabel="EC50 pActivity", ylabel="IC50 pActivity",
             title="Molecules measured both ways")

We pool them by averaging the two pActivity values. For the label we trust the two
endpoint labels first, since each already knows whether it came from a measurement or
a limit, and only fall back to the pooled number when they contradict each other.

In [ ]:
labels = wide[["activity_ec50", "activity_ic50"]]
conflict = labels.notna().sum(axis=1).eq(2) & labels.nunique(axis=1).gt(1)
final = wide.assign(
    pactivity=wide[["pactivity_ec50", "pactivity_ic50"]].mean(axis=1))
final["activity"] = np.where(conflict,
    (final["pactivity"] >= THRESHOLD).astype(int),
    labels.bfill(axis=1).iloc[:, 0]).astype(int)
print(f"{conflict.sum():,} molecules had the two endpoints disagreeing")

Last, attach each molecule's structure and build the final table. It needs the
InChIKey, the SMILES and the label at a minimum, since that is what the next notebook
will read.

In [ ]:
identity = molecules.groupby("inchikey").agg(smiles=("smiles", "first"),
                                            chembl_id=("chembl_id", "first"))
final = identity.join(final)
print(f"{len(final):,} molecules, {final['activity'].mean():.1%} of them active")
final.head(3)

Colab forgets everything when it disconnects, so this table has to leave the
notebook or it is gone. The cell below writes it to a file and, in Colab, starts a
download to your own computer.

Once it has downloaded, upload `hiv1_curated.csv` to the group's Drive folder
**Projects/PurpleTeam/Data**. That is how the rest of the group, and the next
notebook, get hold of it.

In [ ]:
import os

os.makedirs("outputs", exist_ok=True)
output_path = "outputs/hiv1_curated.csv"
final.to_csv(output_path)

if "google.colab" in sys.modules:
    from google.colab import files
    files.download(output_path)
print(f"written to {output_path} ({os.path.getsize(output_path) / 1e6:.1f} MB)")

> **Exercise:** Averaging assumes the two endpoints are equally trustworthy. Look
> again at the plot above: is the disagreement even, or does one endpoint tend to read
> weaker than the other? Try keeping EC50 alone, and compare how many molecules you
> end up with against how consistent they are. Deciding to use less data in exchange
> for cleaner data is a normal thing to do.

## 9. A second dataset for regression

Everything so far has been aimed at a yes-or-no question: is this molecule active or
not. A different and more demanding question is **how** active it is, predicting the
pActivity itself rather than a label. That is called regression, and the limits we
rescued in section 5 stop working there.

The reason is that a limit is not a number. `IC50 > 50 uM` told us reliably that the
molecule is inactive, which was all the label needed. But asking a model to predict
the value 4.3 for that molecule is asking it to learn something nobody measured: the
true value is somewhere weaker than 50 uM, and 50 uM is only where the experiment
stopped. Training on those numbers teaches the model the habits of the laboratories
rather than the chemistry.

So we build a second, smaller table that keeps only real measurements.

In [ ]:
measured_only = molecules[molecules["source"] == "measured"]
regression = measured_only.pivot(index="inchikey", columns="endpoint",
                                 values="pactivity")
regression.columns = [f"pactivity_{name.lower()}" for name in regression.columns]
regression["pactivity"] = regression.mean(axis=1)
regression = identity.join(regression, how="right")
print(f"{len(final):,} molecules for classification")
print(f"{len(regression):,} molecules for regression")

So how many compounds does that cost us? Count them both ways: as a number and as a
share of what we had.

In [ ]:
lost = final.index.difference(regression.index)
print(f"lost {len(lost):,} molecules, {len(lost) / len(final):.1%} of the set")
for endpoint in ["EC50", "IC50"]:
    at_all = molecules.loc[molecules["endpoint"] == endpoint, "inchikey"].nunique()
    real = measured_only.loc[measured_only["endpoint"] == endpoint, "inchikey"].nunique()
    print(f"  {endpoint}: {at_all:,} -> {real:,} ({1 - real / at_all:.1%} lost)")

Losing that many molecules sounds like the whole story, but it is not. What matters
more is **which** molecules we lost, and they are not a random sample. Look at the
labels they carried.

In [ ]:
final.loc[lost, "activity"].value_counts().rename(
    {0: "inactive", 1: "active"}).to_frame("molecules lost")

Almost everything we dropped was inactive, which makes sense: a limit is what you
record when a compound does not work well enough to measure properly. The regression
set is therefore not just smaller, it is **shifted towards the potent compounds**. A
model trained on it has barely seen a weak molecule, so it will be poor at
recognising one.

The plot below shows the two distributions together. The regression set is missing
most of its left-hand side.

In [ ]:
fig, axs = stylia.create_figure(1, 1)
ax = axs.next()
for values, color, name in [(final["pactivity"], nc.purple, "classification"),
                            (regression["pactivity"], nc.mint, "regression")]:
    ax.hist(values, bins=60, range=(2, 12), histtype="stepfilled", alpha=0.6,
            color=color, label=f"{name} (n={len(values):,}, "
                               f"median {values.median():.2f})")
ax.axvline(THRESHOLD, color=nc.pink, linestyle="--")
ax.legend()
stylia.label(ax, xlabel="pActivity", ylabel="Molecules",
             title="What the regression set leaves out")

Download this one as well, next to the first. Keep both: they answer different
questions and neither replaces the other.

In [ ]:
regression_path = "outputs/hiv1_regression.csv"
regression.to_csv(regression_path)

if "google.colab" in sys.modules:
    from google.colab import files
    files.download(regression_path)
print(f"written to {regression_path} "
      f"({os.path.getsize(regression_path) / 1e6:.1f} MB)")

> **Exercise:** A sixth of the molecules is a real price. Decide whether it is worth
> paying, and remember you have a third option: keep the limits but mark them, and
> train a model that knows which values are real and which are only bounds. Then ask
> the harder question. How many of the molecules left in the regression set rest on a
> single measurement, and what does section 6 tell you about how much you should
> trust one number? You may find a stricter set is smaller still.

## Summary

- You turned two raw ChEMBL downloads into one table with a single row per molecule,
  standardising the structures so that salts and duplicates were merged.
- Repeated measurements of the same molecule disagree more than people expect: the
  median spread is around one log unit, meaning a tenfold difference between
  laboratories, and for a sizeable group of molecules that decides their label.
- Limits such as `> 50 uM` are worth keeping, but only when they already settle the
  question at the cutoff you chose.
- You downloaded the result as `hiv1_curated.csv`, with an InChIKey, a SMILES, a
  pActivity and an active/inactive label for each molecule, and put it in the group's
  Drive folder so everyone works from the same table.
- Predicting potency itself needs real numbers, not limits. Dropping them costs about
  a sixth of the molecules, and because almost all of those were inactive, the second
  table is left leaning towards the potent compounds.

**Next:** upload your `hiv1_curated.csv` to the group's Drive folder, then compare
what everyone got. Different rules give different tables, and the differences are
worth discussing before anyone trains a model on them.